In [2]:
import cv2
import numpy as np

# Laden der YOLO-Modelldateien
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")

# Definieren der Klassen, die von YOLO erkannt werden sollen
classes = []
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Öffnen der Webcam
cap = cv2.VideoCapture(0)

while True:
    # Lesen des aktuellen Frames aus der Webcam
    ret, frame = cap.read()

    # Skalieren des Bildes auf die Größe, die vom YOLO-Modell erwartet wird
    blob = cv2.dnn.blobFromImage(frame, 1/255, (416, 416), swapRB=True, crop=False)

    # Setzen des Blob als Eingabe für das YOLO-Modell
    net.setInput(blob)

    # Durchführen der Vorhersage auf dem Modell
    outs = net.forward(net.getUnconnectedOutLayersNames())

    # Verarbeiten der Vorhersagen und Zeichnen der Bounding Boxes auf das Bild
    num_people = 0
    boxes = []
    confidences = []
    class_ids = []
    for out in outs:
        for detection in out:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if class_id == 0 and confidence > 0.5:
                num_people += 1
                x, y, w, h = detection[:4] * np.array([frame.shape[1], frame.shape[0], frame.shape[1], frame.shape[0]])
                boxes.append([x-w/2, y-h/2, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Anwenden der Non-Maximum Suppression
    indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

    # Zeichnen der verbleibenden Bounding Boxes auf das Bild und Ausgabe der Anzahl der Personen
    for i in indices:
        i = i[0]
        x, y, w, h = boxes[i]
        cv2.rectangle(frame, (int(x), int(y)), (int(x+w), int(y+h)), (0,255,0), 2)
    cv2.imshow("Webcam", frame)
    print("Anzahl Personen: ", len(indices))

    # Abbrechen der Schleife bei Drücken der 'q'-Taste
    if cv2.waitKey(1) == ord('q'):
        break

# Freigeben der Ressourcen
cap.release()
cv2.destroyAllWindows()

IndexError: invalid index to scalar variable.